# Busca de dados - Copa Aluna

Use este notebook para coletar os dados da Copa Aluna (16 times) e gerar os arquivos em `datasets_copa_aluna`.


In [1]:
import cartolafc
import pandas as pd
import json
import time
import requests
from functools import lru_cache

pd.set_option('display.max_columns', 50)            # permite a visualizacao de 50 colunas do dataframe
pd.options.display.float_format = '{:.2f}'.format   # pandas: para todos os numeros aparecerem com duas casas decimais

# Cria uma instancia da API
api = cartolafc.Api(attempts=5)

# Constantes do 1- turno
INICIO_COPA = 16
FIM_COPA = 19
COLUNAS_RODADAS = [f"Rodada {r}" for r in range(INICIO_COPA, FIM_COPA + 1)]

2026-06-01 13:25:32,154 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


In [2]:
# Lista fixa de IDs dos participantes (lista de int)
ids_participantes = [
    19833277, 1488983,
    287965, 20651178,
    1273719, 1867254,
    2916559, 186283,
    1747619, 47775950,
    32966, 184499,
    19209079, 14933455,
    16411206, 44810918,       
     
    # 4088673, 1326835, 2371918, 14709358
]

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json,text/plain,*/*",
    "Referer": "https://cartola.globo.com/",
}

@lru_cache(maxsize=5000)
def nome_time_por_id_api(time_id: int, timeout=15) -> str:
    endpoints = [
        f"https://api.cartolafc.globo.com/time/id/{time_id}",
        f"https://api.cartolafc.globo.com/time/{time_id}",
    ]

    for url in endpoints:
        for tentativa in range(3):
            try:
                r = requests.get(url, headers=HEADERS, timeout=timeout)
                if r.status_code == 429:
                    time.sleep(0.8 + tentativa * 0.8)
                    continue
                if r.status_code != 200:
                    break

                data = r.json()
                if isinstance(data, dict):
                    if isinstance(data.get("time"), dict) and isinstance(data["time"].get("nome"), str):
                        return data["time"]["nome"]
                    if isinstance(data.get("nome"), str):
                        return data["nome"]
                break
            except Exception:
                time.sleep(0.5)
                continue

    return f"Time {time_id}"

In [4]:
# Base com todos os participantes
if not isinstance(ids_participantes, list) or not ids_participantes:
    raise ValueError("ids_participantes precisa ser uma lista de IDs")

ids_participantes = list(dict.fromkeys(ids_participantes))

df_base = pd.DataFrame({"time_id": ids_participantes}).drop_duplicates()
df_base["Time"] = df_base["time_id"].apply(nome_time_por_id_api)

df_base = df_base.set_index("time_id").sort_index()

# Dicionario Nome -> ID (compatibilidade com codigo legado)
ids_times_dict = {row["Time"]: row["time_id"] for _, row in df_base.reset_index().iterrows()}

# Links para o Excel
df_urls = pd.DataFrame({
    "Nome do Time": df_base["Time"].values,
    "ID do Time": df_base.index.values,
})

df_urls["Link do Time"] = df_urls["ID do Time"].apply(
    lambda x: f"https://cartola.globo.com/#!/time/{x}"
)

df_urls = df_urls[["Nome do Time", "ID do Time", "Link do Time"]]

caminho_excel = "links_times_cartola_copa_aluna.xlsx"
df_urls.to_excel(caminho_excel, index=False)
print(f"- Arquivo salvo com sucesso: {caminho_excel}")

display(df_base)
# display(df_urls)

- Arquivo salvo com sucesso: links_times_cartola_copa_aluna.xlsx


,Time
time_id,
32966,La Primeira Patada Es Nuestra
184499,SC ÉoINTER!
186283,FBC Colorado
287965,Doug Leal F.C
1273719,Texas Club 2026
1488983,C R Juvenal
1747619,JV5 Tricolor Gaúcho
1867254,Medonho´s F.C.
2916559,Esquadrão Gazembrino


In [5]:
def campeonato_comecou(ids, rodada_ref=INICIO_COPA):
    lista_ids = list(ids.values()) if isinstance(ids, dict) else list(ids)
    for time_id in lista_ids:
        try:
            t = api.time(time_id=time_id, rodada=rodada_ref)
            v = getattr(t, "ultima_pontuacao", None)
            if v is not None:
                return True
        except Exception:
            continue
    return False

In [6]:
# GERAR copa_dados_aluna.js
import json
from pathlib import Path

FASES_POR_RODADA = {
    INICIO_COPA: "oitavas",
    INICIO_COPA + 1: "quartas",
    INICIO_COPA + 2: "semi",
    INICIO_COPA + 3: "final",
}

# Mapa id -> nome (a partir do df_base)
mapa_nomes = df_base.reset_index().set_index('time_id')['Time'].to_dict()

mercado = api.mercado()
rodada_atual = int(getattr(mercado, 'rodada_atual', INICIO_COPA))
status_mercado = getattr(getattr(mercado, 'status', None), 'id', None)
RODADA_MAX_COM_PONTOS = rodada_atual - 1 if status_mercado == 1 else rodada_atual
RODADA_MAX_COM_PONTOS = max(INICIO_COPA, min(FIM_COPA, RODADA_MAX_COM_PONTOS))


def _normalizar_nome(nome):
    import re
    import unicodedata

    texto = str(nome or "").strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())

def obter_pontuacao_rodada(time_id, rodada=INICIO_COPA):
    if int(rodada) > int(RODADA_MAX_COM_PONTOS):
        return None
    try:
        t = api.time(time_id=int(time_id), rodada=int(rodada))
        pts = getattr(t, "ultima_pontuacao", None)
        if pts is not None:
            return float(pts)
    except Exception:
        pass

    try:
        liga_path = Path('../../liga_classica_aluna/data/Pontuacoes_Liga_Classica_Aluna.js')
        conteudo = liga_path.read_text(encoding='utf-8')
        trecho = conteudo.split('const classificacaoLigaClassica = ', 1)[1].split('};window.ligaClassicaMeta', 1)[0] + '}'
        classificacao = json.loads(trecho)
        geral = classificacao.get('geral', {})
        mapa_normalizado = {_normalizar_nome(nome): dados for nome, dados in geral.items()}
        nome = mapa_nomes.get(int(time_id), f'Time {time_id}')
        dados = mapa_normalizado.get(_normalizar_nome(nome), {})
        valor = dados.get(f'Rodada {int(rodada)}')
        return float(valor) if valor is not None else None
    except Exception:
        return None

def obter_total_turno(time_id, rodada_limite=INICIO_COPA):
    try:
        liga_path = Path('../../liga_classica_aluna/data/Pontuacoes_Liga_Classica_Aluna.js')
        conteudo = liga_path.read_text(encoding='utf-8')
        trecho = conteudo.split('const classificacaoLigaClassica = ', 1)[1].split('};window.ligaClassicaMeta', 1)[0] + '}'
        classificacao = json.loads(trecho)
        geral = classificacao.get('geral', {})
        mapa_normalizado = {_normalizar_nome(nome): dados for nome, dados in geral.items()}
        nome = mapa_nomes.get(int(time_id), f'Time {time_id}')
        dados = mapa_normalizado.get(_normalizar_nome(nome), {})
        total = sum(
            float(v)
            for k, v in dados.items()
            if str(k).startswith('Rodada ') and int(str(k).split()[1]) <= int(rodada_limite)
        )
        return float(total) if dados else None
    except Exception:
        return None

def _vencedor_match(match):
    casa_id = match.get('casaId')
    fora_id = match.get('foraId')
    casa_pts = match.get('casaPts')
    fora_pts = match.get('foraPts')
    if casa_id is None or fora_id is None:
        return None
    if casa_pts is None and fora_pts is None:
        return None
    casa_val = float(casa_pts) if casa_pts is not None else float('-inf')
    fora_val = float(fora_pts) if fora_pts is not None else float('-inf')
    return casa_id if casa_val >= fora_val else fora_id

def _perdedor_match(match):
    vencedor = _vencedor_match(match)
    if vencedor is None:
        return None
    return match.get('foraId') if vencedor == match.get('casaId') else match.get('casaId')

def _montar_fase_por_ids(ids_fase, rodada_fase):
    if len(ids_fase) < 2:
        return []
    fase = []
    for idx in range(0, len(ids_fase) - 1, 2):
        casa_id = int(ids_fase[idx])
        fora_id = int(ids_fase[idx + 1])
        fase.append({
            'casaId': casa_id,
            'foraId': fora_id,
            'casaPts': obter_pontuacao_rodada(casa_id, rodada_fase),
            'foraPts': obter_pontuacao_rodada(fora_id, rodada_fase),
        })
    return fase

# Times (ordem conforme ids_participantes)
times = [
    {
        'id': int(time_id),
        'nome': mapa_nomes.get(int(time_id), f'Time {time_id}'),
        'turnoPts': obter_total_turno(int(time_id), FIM_COPA),
    }
    for time_id in ids_participantes
]

# Confrontos das oitavas (chave fixa)
confrontos_oitavas = [
    (19833277, 1488983),
    (287965, 20651178),
    (1273719, 1867254),
    (2916559, 186283),
    (1747619, 47775950),
    (32966, 184499),
    (19209079, 14933455),
    (16411206, 44810918),
]

oitavas = [
    {
        'casaId': casa,
        'foraId': fora,
        'casaPts': obter_pontuacao_rodada(casa, INICIO_COPA),
        'foraPts': obter_pontuacao_rodada(fora, INICIO_COPA),
    }
    for casa, fora in confrontos_oitavas
]

vencedores_oitavas = [_vencedor_match(match) for match in oitavas]
vencedores_oitavas = [item for item in vencedores_oitavas if item is not None]
quartas = _montar_fase_por_ids(vencedores_oitavas, INICIO_COPA + 1)

vencedores_quartas = [_vencedor_match(match) for match in quartas]
vencedores_quartas = [item for item in vencedores_quartas if item is not None]
semi = _montar_fase_por_ids(vencedores_quartas, INICIO_COPA + 2)

vencedores_semi = [_vencedor_match(match) for match in semi]
perdedores_semi = [_perdedor_match(match) for match in semi]
vencedores_semi = [item for item in vencedores_semi if item is not None]
perdedores_semi = [item for item in perdedores_semi if item is not None]

final = _montar_fase_por_ids(vencedores_semi, FIM_COPA)
if not final and len(vencedores_semi) == 2:
    final = [{'casaId': vencedores_semi[0], 'foraId': vencedores_semi[1], 'casaPts': None, 'foraPts': None}]

terceiro = _montar_fase_por_ids(perdedores_semi, FIM_COPA)
if not terceiro and len(perdedores_semi) == 2:
    terceiro = [{'casaId': perdedores_semi[0], 'foraId': perdedores_semi[1], 'casaPts': None, 'foraPts': None}]

pontuacoes_por_fase = {
    fase: {
        str(int(time_id)): obter_pontuacao_rodada(int(time_id), rodada)
        for time_id in ids_participantes
    }
    for rodada, fase in FASES_POR_RODADA.items()
}

copa_dados = {
    'temporada': '2026/1',
    'times': times,
    'pontuacoes_por_fase': pontuacoes_por_fase,
    'fases': {
        'oitavas': oitavas,
        'quartas': quartas,
        'semi': semi,
        'final': final,
        'terceiro': terceiro,
    },
}

saida = Path('copa_dados_aluna.js')
conteudo_js = 'window.copaDados = ' + json.dumps(copa_dados, ensure_ascii=False, indent=2) + ';'
saida.write_text(conteudo_js, encoding='utf-8')
print(f'Arquivo gerado: {saida.resolve()}')


Arquivo gerado: C:\Users\ferna\Projetos\GitHub\cartola_2026\copa_aluna\datasets_copa_aluna\copa_dados_aluna.js
